In [0]:
dbutils.widgets.text("environment_name", "")

In [0]:
env_name = dbutils.widgets.get("environment_name")
print(f"environment name is {env_name}")

In [0]:
from pyspark.sql.functions import year, col, sum as _sum, count, round as _round

silver_df = spark.table(f"{env_name}_silver.orders")

orders_gold_df = (
    silver_df
        .withColumn("order_year", year(col("o_orderdate")))
        .groupBy("order_year", "o_orderstatus")
        .agg(
            count("o_orderkey").alias("order_count"),
            _round(_sum("o_totalprice"), 2).alias("total_revenue")
        )
        .orderBy("order_year", "o_orderstatus")
)

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {env_name}_gold")

(
    orders_gold_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{env_name}_gold.orders_summary")
)

In [0]:
display(spark.sql(f"SELECT * FROM {env_name}_gold.orders_summary ORDER BY order_year, o_orderstatus"))